# Flow preprocessing

Clean Hilo flow CSVs from `downloaded_data/processed_data`, obfuscate IPs, map `userclass` from the IP maps, and encode features for classification.

Taken from `hilo_flow_data_11_days.ipynb`. Clustering (K-Means, PCA, t-SNE) is not included.


## Data Clean

In [ ]:
%pip install git+https://github.com/jedisct1/draft-denis-ipcrypt.git#subdirectory=implementations/python

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import os
import glob
from ipcrypt import deterministic # Deterministic Encryption
import ipaddress
from sklearn.preprocessing import LabelEncoder


In [ ]:
path = "downloaded_data/processed_data"
csv_files = glob.glob(os.path.join(path, "*.csv"))
df = pd.concat((pd.read_csv(file) for file in csv_files), ignore_index=True)

df

In [ ]:
df = df.replace('-', np.nan)
df = df.dropna()

In [ ]:
print(df['dstIP'].nunique()) # => 442
print(df['srcIP'].nunique()) # => 94819

#### Before dropping 4th octet
- df['dstIP'].nunique() # => 289
- df['srcIP'].nunique() # => 9222

### IP Address to User Type Mapping

In [ ]:
ip_matches = pd.read_csv("ip_user_map-v1.csv")
ip_matches_2 = pd.read_csv("HiloFlowIPMissingMap.csv")
ip_matches_2 = ip_matches_2.drop(columns=["hostname", "Who to ask"]).dropna()

ip_matches = ip_matches.rename(columns={
    "ipaddr": "missing_srcIP",
    "userclass": "UserClass"
})
ip_matches["missing_srcIP"] = ip_matches["missing_srcIP"].str.strip()
ip_matches_2["missing_srcIP"] = ip_matches_2["missing_srcIP"].str.strip()

# ip_matches.nunique()

In [ ]:
ip_matches_concat = pd.concat([ip_matches,ip_matches_2], ignore_index=True).drop_duplicates()
ip_matches_concat

In [ ]:
df['srcIP'] = df['srcIP'].astype(str).str.strip()
df['dstIP'] = df['dstIP'].astype(str).str.strip()
df.info()

### IP Crypt Deterministic IP Obfuscation (New Approach)

In [ ]:
key = bytes.fromhex("0123456789abcdeffedcba9876543210")


df['srcIP_obfuscated'] = df['srcIP'].apply(lambda x: deterministic.encrypt(x, key))
df['dstIP_obfuscated'] = df['dstIP'].apply(lambda x: deterministic.encrypt(x, key))

In [ ]:
df['srcIP_obfuscated'] = df['srcIP_obfuscated'].apply(lambda x: int(ipaddress.IPv6Address(x)))

In [ ]:
df.info()

### Mapping userclass to srcIPs

In [ ]:
df.info()

In [ ]:
ip_matches_concat.info()

In [ ]:
ip_to_userclass = dict(zip(ip_matches_concat["missing_srcIP"], ip_matches_concat["UserClass"]))

# print(ip_to_userclass)
# map userclass to srcIP
df["userclass"] = df["srcIP"].map(ip_to_userclass)
df.head()

In [ ]:
df_userclass = df[df["userclass"].notna()]
# df_userclass.info()
df_userclass["userclass"].value_counts()
df_userclass

df_userclass.to_csv("userclasses_mapped_v2.csv", index=False)

In [ ]:
# df.head(60)
combined_ips = set(df["srcIP"]).union(set(df["dstIP"]))
matching_ips = combined_ips.intersection(set(ip_matches_concat["missing_srcIP"]))

missing_src_ips = df.loc[~df["srcIP"].isin(ip_matches_concat["missing_srcIP"]), "srcIP"].unique()
num_unique_src_ips = df["srcIP"].nunique()
num_matching_ips = len(matching_ips)
print("df unique ips:", num_unique_src_ips)
print("matching ips:", num_matching_ips)
print("missing src ips", missing_src_ips)

### Converts IPs from strings to floats

In [ ]:
import ipaddress
# Bitwise operation shift to turn IPs into float dtype
def ip_to_float(ip):
    try:
        return float(int(ipaddress.ip_address(ip)))
    except:
        return None
        
# df['masked_srcIP'] = df['masked_srcIP'].apply(ip_to_float)
# df['masked_dstIP'] = df['masked_dstIP'].apply(ip_to_float)

In [ ]:
# Convert tcpFlags from hexadecimal => int64 values
# df['tcpFlags'] = df['tcpFlags'].apply(lambda x : int(x, 16)).astype('uint8')

In [ ]:
df.info()

In [ ]:
# df['dstIP'].nunique() => 15
# df['srcIP'].nunique() => 4271
# df['srcMAC'].nunique() # => 64 unique devices potentially

In [ ]:
# Total number of unique interfaces
unique_macs = pd.concat([df['srcMAC'], df['dstMAC']]).nunique()
unique_macs

In [ ]:
# Keeping necessary columns for analysis & model training
keep_cols = ['srcIP_obfuscated', 'dstIP_obfuscated', 'srcPort', 'dstPort', 'ipProtocol', 'tcpFlags', 'packetSize', 'payloadSize', 'ipTTL']
# keep_cols = ['srcPort', 'dstPort', 'ipProtocol', 'tcpFlags', 'packetSize', 'payloadSize', 'ipTTL']

new_df = df[keep_cols]
new_df

In [ ]:
# Convert tcpFlags from hexadecimal => int64 values
# new_df['tcpFlags'] = new_df['tcpFlags'].apply(lambda x : int(x, 16)).astype('uint8')

In [ ]:
new_df.info()

In [ ]:
# Convert Object dtypes to labels
categorical_cols = new_df.select_dtypes(include=['object']).columns.tolist()

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    new_df[col] = le.fit_transform(new_df[col].astype(str))
    label_encoders[col] = le

In [ ]:
new_df = new_df.dropna()

# new_df.info()
new_df.shape

In [ ]:
new_df.info()